# 07-01 正则化方法深入理解：怎么让模型别只背训练集

上一节我们讲了泛化能力和训练曲线诊断。

现在进入正则化。

正则化不是为了让训练集表现更好，而是为了让模型在新数据上更可靠。

它要解决的问题是：

```text
模型太有能力了，可能把训练集里的偶然细节也记住。
```

所以正则化的核心思想是：

```text
让模型学习规律，但不要让它用太极端、太复杂、太依赖局部细节的方式学习。
```

## 1. 正则化先别想成某一个公式

很多资料一讲正则化就直接写：

$$
\mathcal{L}_{total}=\mathcal{L}_{train}+\lambda\Omega(\theta)
$$

这个公式没错，但初学者容易看完就断掉。

我们先把它翻译成人话。

普通训练只关心一件事：

```text
训练集上错得越少越好。
```

正则化会补一句：

```text
你可以降低训练错误，但别用太夸张的方式去降低。
```

所以总损失变成两部分：

$$
\mathcal{L}_{total}
=
\mathcal{L}_{train}
+
\lambda\Omega(\theta)
$$

第一部分管训练误差。

第二部分管模型复杂度。

## 2. lambda 是什么

公式里有一个很关键的系数：

$$
\lambda
$$

它控制正则化强度。

如果：

$$
\lambda=0
$$

那正则化项就没作用，模型只管训练集 loss。

如果：

$$
\lambda\text{ 很大}
$$

模型会非常害怕复杂，甚至连训练集都学不好。

所以 `lambda` 不是越大越好。

它像一个约束力度：

```text
太小：管不住模型，可能过拟合
太大：管得太狠，可能欠拟合
合适：牺牲一点训练集表现，换更好的泛化能力
```

## 3. L2 正则化：不要让权重太大

L2 正则化最常见，也经常叫权重衰减。

它的惩罚项是：

$$
\Omega(\theta)=\sum_i w_i^2
$$

加到总损失里就是：

$$
\mathcal{L}_{total}
=
\mathcal{L}_{train}
+
\lambda\sum_i w_i^2
$$

这句话的意思是：

```text
权重越大，模型要付出的惩罚越大。
```

所以训练时，模型会尽量避免把权重调得特别大。

## 4. 为什么大权重容易出问题

先看一个最简单的线性关系：

$$
y=wx+b
$$

如果权重很大，比如：

$$
w=100
$$

那么输入稍微变一点，输出就会变很多。

例如输入变化：

$$
\Delta x=0.01
$$

输出变化大约是：

$$
\Delta y=w\Delta x=100\times0.01=1
$$

输入只动了一点点，输出却明显变化。

这说明模型对输入细节很敏感。

在神经网络里，过大的权重可能让模型为了适应训练集中某几个特殊样本，把决策边界扭得很复杂。

L2 正则化就是在提醒模型：

```text
不要为了训练集里的少数细节，把参数调得太极端。
```

## 5. L2 为什么叫权重衰减

我们看一下直觉，不做复杂推导。

普通梯度下降更新是：

$$
w_t=w_{t-1}-\eta\frac{\partial\mathcal{L}_{train}}{\partial w}
$$

加入 L2 正则后，总损失是：

$$
\mathcal{L}_{total}
=
\mathcal{L}_{train}
+
\lambda w^2
$$

对权重求梯度时，正则项会贡献：

$$
\frac{\partial}{\partial w}\lambda w^2=2\lambda w
$$

所以更新里会多出一个把权重往小拉的力量：

$$
w_t
=
w_{t-1}
-
\eta
\left(
\frac{\partial\mathcal{L}_{train}}{\partial w}
+
2\lambda w
\right)
$$

这就是为什么它叫权重衰减。

它会不断给权重一个“别太大”的压力。

## 6. L1 正则化：让一些权重变成 0

L1 正则化的惩罚项是：

$$
\Omega(\theta)=\sum_i |w_i|
$$

总损失写成：

$$
\mathcal{L}_{total}
=
\mathcal{L}_{train}
+
\lambda\sum_i |w_i|
$$

L1 的一个重要特点是：它更容易让某些权重变成 0。

权重变成 0，意思是对应输入特征对模型几乎不起作用。

所以 L1 常和“稀疏”联系在一起。

可以这样记：

```text
L2：更像让权重整体变小
L1：更像让一部分权重直接变成 0
```

## 7. Dropout：训练时随机关闭一部分神经元

Dropout 的做法是：训练时随机让一部分神经元暂时不工作。

假设某一层输出是：

$$
\mathbf{h}=[h_1,h_2,h_3,h_4]
$$

Dropout 可能随机变成：

$$
\tilde{\mathbf{h}}=[h_1,0,h_3,0]
$$

注意，这不是说第二个和第四个神经元没用。

而是训练时故意制造一种情况：

```text
你不能总依赖某几个固定神经元。
```

这样模型会被迫学习更分散、更稳定的表示。

## 8. Dropout 为什么能缓解过拟合

没有 Dropout 时，网络可能形成很强的依赖关系。

比如某个神经元专门记住训练集里的一个特殊细节，后面的层也特别依赖它。

这会让模型在训练集上很强，但对新数据很脆弱。

Dropout 随机关闭神经元，相当于不断打乱这种依赖：

```text
今天这个神经元可能不在，明天另一个神经元可能不在。
模型不能只靠单一路径解决问题。
```

所以它会鼓励网络学到更稳健的特征组合。

一句话记：

```text
Dropout 通过随机断开依赖，减少模型对训练集局部细节的记忆。
```

## 9. Dropout 训练和测试不一样

Dropout 通常只在训练阶段使用。

训练时：

```text
随机关闭一部分神经元
```

测试时：

```text
所有神经元都正常工作
```

为什么测试时不继续随机关闭？

因为测试时我们希望模型稳定输出，而不是每次预测都随机变。

训练阶段 Dropout 是为了逼模型变稳健。

测试阶段关闭 Dropout，是为了使用完整模型做确定预测。

## 10. Early Stopping：不要训练到背题

早停的想法很简单。

训练过程中，我们同时观察训练集和验证集。

如果出现：

$$
\mathcal{L}_{train}\downarrow
$$

但是：

$$
\mathcal{L}_{val}\uparrow
$$

就说明模型可能开始过拟合。

Early Stopping 会在验证集表现最好的时候保存模型。

它的核心不是“少训练”，而是：

```text
训练到泛化能力最好的位置就停下来。
```

## 11. Data Augmentation：从数据上减少背题

数据增强的思路是：在不改变标签的前提下，制造更多合理变化。

比如图像分类中，一张猫的图片：

```text
轻微裁剪后还是猫
左右翻转后还是猫
亮度稍微变化后还是猫
```

这些变化可以让模型看到更多情况。

这样模型就不容易只记住：

```text
猫必须出现在图片正中间
背景必须是某种颜色
光照必须和训练图片一样
```

数据增强是在告诉模型：

```text
真正重要的是稳定特征，不是训练图片里的固定细节。
```

## 12. 控制模型复杂度

如果模型太复杂，也可以直接降低模型能力。

比如：

```text
减少隐藏层数量
减少每层神经元数量
减少参数量
使用更简单的结构
```

这不是说模型越简单越好。

模型太简单会欠拟合。

模型太复杂又容易过拟合。

关键是匹配：

```text
数据量越少，越要小心模型太复杂。
任务越复杂，模型又需要足够表达能力。
```

所以模型复杂度要和数据量、任务难度一起看。

## 13. 正则化不是越多越好

正则化的目标是缓解过拟合，但它也会限制模型学习能力。

如果正则化太弱，模型可能还是过拟合。

如果正则化太强，模型可能变成欠拟合。

可以这样看：

| 情况 | 可能问题 | 调整方向 |
| --- | --- | --- |
| 训练好，验证差 | 过拟合 | 加强正则化 |
| 训练差，验证也差 | 欠拟合 | 减弱正则化或增强模型 |
| 训练和验证都逐渐变好 | 正常学习 | 先不乱改 |

所以判断是否要加正则化，一定要先看训练曲线。

## 14. 常见正则化方法怎么记

| 方法 | 它在限制什么 | 一句话记法 |
| --- | --- | --- |
| L2 / 权重衰减 | 限制权重太大 | 别把参数调得太极端 |
| L1 | 让部分权重变成 0 | 不重要的特征就别用 |
| Dropout | 限制神经元依赖 | 别总靠固定几个神经元 |
| Early Stopping | 限制训练时间 | 到验证集最好时停 |
| Data Augmentation | 限制对固定样本的记忆 | 多看合理变化 |
| 降低模型复杂度 | 限制表达能力 | 别给模型太强的背题能力 |

这些方法不是互斥的。

实际训练中，经常会组合使用。

## 15. 本节总结

正则化的逻辑链是：

```text
模型太复杂时，可能记住训练集细节
-> 训练集表现好，不代表新数据表现好
-> 正则化给模型加约束
-> L2 限制权重太大
-> L1 鼓励部分权重变成 0
-> Dropout 减少神经元之间的固定依赖
-> Early Stopping 防止训练太久
-> Data Augmentation 让模型见到更多合理变化
```

先记住一句话：

```text
正则化不是为了让训练题分数更高，而是为了让模型别只会做训练题。
```

下一节适合继续讲 Batch Normalization：它不是典型正则化，但它能让深层网络训练更稳定。